# Week 1 lecture walkthrough: from matching summaries to constructed shape

This is the worked example for live teaching. It follows the conceptual sequence of the Week 1 slides and makes each computational claim visible. It is not an answer key to every practical choice.

**Resource boundary.** The [reference notes](index.qmd) explain the ideas fully. The [slides](slides.qmd) carry the visual argument. This notebook supplies the lecturer's prediction-and-reveal sequence. The [participant practical](lab.ipynb) asks students to make their own descriptor, threshold and interpretation choices.

**Lecture map.** First reveal the 13 Datasaurus clouds after their matching summaries. Then compare geometric descriptors, test which transformations retain distances, and finally keep the observations fixed while changing a graph threshold. The conclusion is about selective summaries and constructed representations, not yet persistent homology.

Suggested rhythm: show the slide, ask for a prediction, then reveal the corresponding notebook output.



## Demonstration setup

The helper functions keep the machinery visible: whitening, pairwise distances, threshold adjacency, component search and the graph identity $E-V+C$. No TDA library is called.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(3024)

def standardise_shape(points):
    """Centre and whiten a two-dimensional point cloud."""
    centred = np.asarray(points, dtype=float) - np.mean(points, axis=0)
    covariance = np.cov(centred, rowvar=False)
    values, vectors = np.linalg.eigh(covariance)
    return centred @ vectors @ np.diag(values ** -0.5)

def scalar_summary(points):
    return {
        'mean_x': np.mean(points[:, 0]),
        'mean_y': np.mean(points[:, 1]),
        'sd_x': np.std(points[:, 0], ddof=1),
        'sd_y': np.std(points[:, 1], ddof=1),
        'correlation': np.corrcoef(points.T)[0, 1],
    }

def pairwise_distances(points):
    differences = points[:, None, :] - points[None, :, :]
    return np.sqrt(np.sum(differences ** 2, axis=2))

def threshold_graph_statistics(points, epsilon):
    """Return V, E, components and graph cycle rank at one threshold."""
    distances = pairwise_distances(points)
    adjacency = (distances <= epsilon) & (distances > 0)
    vertices = len(points)
    edges = int(np.sum(adjacency) // 2)
    unseen = set(range(vertices))
    components = 0
    while unseen:
        components += 1
        stack = [unseen.pop()]
        while stack:
            vertex = stack.pop()
            neighbours = set(np.flatnonzero(adjacency[vertex])) & unseen
            unseen -= neighbours
            stack.extend(neighbours)
    cycle_rank = edges - vertices + components
    return {'vertices': vertices, 'edges': edges,
            'components': components, 'graph_cycle_rank': cycle_rank}

print('Environment ready. Random seed: 3024')

## Demonstration 1, after slide “Matching moments, different organisation”

**Ask before running:** If 13 clouds have nearly identical coordinate means, standard deviations and correlations, how different could their organisation be? What evidence do your eyes use to distinguish them?

Demonstration 1 uses the canonical 1,846-row Datasaurus Dozen CSV. Reveal the table of coordinate moments and correlations first, then reveal the 13 scatter plots.

In [ ]:
from pathlib import Path
import pandas as pd

def find_datasaurus_csv():
    candidates = [
        Path('datasaurus_dozen.csv'),
        Path('data/datasaurus_dozen.csv'),
        Path('../../data/datasaurus_dozen.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        'Download datasaurus_dozen.csv from the Week 1 page and place it '
        'beside this notebook.'
    )

datasaurus_path = find_datasaurus_csv()
datasaurus = pd.read_csv(datasaurus_path)
dataset_order = list(datasaurus['dataset'].drop_duplicates())

groups = datasaurus.groupby('dataset', sort=False)
summaries = groups.agg(
    n=('x', 'size'),
    mean_x=('x', 'mean'),
    mean_y=('y', 'mean'),
    sd_x=('x', 'std'),
    sd_y=('y', 'std'),
)
summaries['correlation'] = [group['x'].corr(group['y']) for _, group in groups]
print(summaries.round(3).to_string())

fig, axes = plt.subplots(4, 4, figsize=(10, 10), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, name in zip(axes.flat, dataset_order):
    group = datasaurus[datasaurus['dataset'] == name]
    ax.scatter(group['x'], group['y'], s=10)
    ax.set_title(name.replace('_', ' '))
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
for ax in axes.flat[len(dataset_order):]:
    ax.axis('off')
fig.supxlabel('x'); fig.supylabel('y')
plt.show()

# A few transparent geometric questions, still without topology.
def convex_hull(points):
    ordered = sorted(set(map(tuple, points)))
    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    lower, upper = [], []
    for point in ordered:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], point) <= 0:
            lower.pop()
        lower.append(point)
    for point in reversed(ordered):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], point) <= 0:
            upper.pop()
        upper.append(point)
    return np.asarray(lower[:-1] + upper[:-1])

def geometric_descriptors(points):
    distances = pairwise_distances(points)
    np.fill_diagonal(distances, np.inf)
    hull = convex_hull(points)
    area = 0.5 * abs(np.dot(hull[:, 0], np.roll(hull[:, 1], 1))
                     - np.dot(hull[:, 1], np.roll(hull[:, 0], 1)))
    radii = np.linalg.norm(points - points.mean(axis=0), axis=1)
    eigenvalues = np.linalg.eigvalsh(np.cov(points.T))
    return {
        'median nearest distance': np.median(distances.min(axis=1)),
        'convex-hull area': area,
        'radial coefficient of variation': radii.std() / radii.mean(),
        'covariance anisotropy': eigenvalues[-1] / eigenvalues[0],
    }

worked_names = ['circle', 'bullseye', 'dots', 'dino']
worked_geometry = {}
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, name in zip(axes, worked_names):
    points = datasaurus.loc[datasaurus['dataset'] == name, ['x', 'y']].to_numpy()
    worked_geometry[name] = geometric_descriptors(points)
    hull = convex_hull(points)
    closed_hull = np.vstack((hull, hull[0]))
    ax.scatter(points[:, 0], points[:, 1], s=9)
    ax.plot(closed_hull[:, 0], closed_hull[:, 1], color='tab:orange', linewidth=1.3)
    ax.set(title=name, xlim=(0, 100), ylim=(0, 100), aspect='equal')
plt.show()
display(pd.DataFrame(worked_geometry).T.round(3))

# Retain one canonical circle for the controlled transformation section.
ring_raw = datasaurus.loc[datasaurus['dataset'] == 'circle', ['x', 'y']].to_numpy()
ring = standardise_shape(ring_raw)


### Teaching point: visual intuition is doing real work

The 13 groups have nearly equal coordinate means, coordinate standard deviations and Pearson correlations, but differ radically in point arrangement. Matejka and Fitzmaurice constructed the datasets through simulated annealing to preserve those numerical values while changing the plotted form. The example motivates richer questions; it does not show that geometry has failed or that topology is always the right description.

First ask the room to contrast `circle`, `bullseye`, `dots`, `star` and `dino`. Collect the visual cues before naming methods: local density, spacing, direction, curvature, boundary, symmetry, clusters, branches, empty regions and scale. Visual judgement combines far more spatial information than the table of coordinate means, coordinate standard deviations and Pearson correlations.

Then separate four questions:

1. **Global summaries:** where is the cloud centred and how do coordinates co-vary?
2. **Richer geometry:** how far apart, dense, curved, directed or symmetric are the observations? A nearest-neighbour distribution, convex hull, density estimate or full distance matrix can distinguish point arrangements that coordinate means, coordinate standard deviations and Pearson correlation cannot distinguish.
3. **Topology and homology:** after constructing a space, how is it connected and which holes occur? Exact lengths and curvature are deliberately ignored.
4. **Persistent topology:** when do selected homological features appear and disappear as the construction scale changes?

Use circle versus ellipse for different geometry but the same topology; circle versus figure eight for different loop structure; and ring boundary versus filled disk for a hole versus no hole. Note that homology is coarse: equal Betti numbers do not imply homeomorphism.

Keep interpretations of the Datasaurus plots provisional. A visible white region is not yet a homology class, and ordinary homology does not completely describe branching. Demonstrations 3 to 5 introduce neighbourhood graphs and scale dependence. Weeks 2 to 4 supply simplicial complexes, homology and persistence. Demonstration 2 uses the canonical `circle` dataset for a controlled transformation comparison.

## Demonstration 2, after slides “Geometry and topology” and “The rubber-sheet idea”

**Ask before running:** Which transformations preserve the pairwise distance matrix? Which preserve the topology of the underlying continuous circle?

Translation and rotation are Euclidean isometries. The anisotropic stretch is still an invertible continuous deformation, but is not an isometry.

In [ ]:
angle = np.deg2rad(35)
rotation = np.array([[np.cos(angle), -np.sin(angle)],
                     [np.sin(angle),  np.cos(angle)]])
transformed = {
    'original': ring,
    'translated': ring + np.array([3.0, -1.5]),
    'rotated': ring @ rotation.T,
    'anisotropically stretched': ring @ np.diag([1.8, 0.55]),
}

baseline = pairwise_distances(ring)
for name, points in transformed.items():
    change = np.max(np.abs(pairwise_distances(points) - baseline))
    print(f'{name:26s} maximum distance change = {change:.3f}')

### Teaching point

All three transformed continuous rings are homeomorphic to the original circle. Only translation and rotation leave every Euclidean pairwise distance unchanged. A construction based on one fixed metric threshold can therefore change under a homeomorphism.

**▶ Likely sticking point.** Topological invariance of an underlying continuous space does not make every data-derived metric construction invariant.

## Demonstration 3, after slide “Finite data do not arrive with a hole label”

The noisy ring and disk are finite samples. Neither comes labelled with the topology of a latent space.

**Ask before running:** What modelling object could we build using only pairwise distances, and which choice would it require?

In [ ]:
n_noisy = 90
angles = RNG.uniform(0, 2 * np.pi, n_noisy)
noisy_ring = np.column_stack((np.cos(angles), np.sin(angles)))
noisy_ring += RNG.normal(0, 0.055, size=(n_noisy, 2))

disk_radii = np.sqrt(RNG.uniform(0, 1, n_noisy))
disk_angles = RNG.uniform(0, 2 * np.pi, n_noisy)
noisy_disk = np.column_stack((disk_radii * np.cos(disk_angles),
                              disk_radii * np.sin(disk_angles)))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.7), constrained_layout=True)
for ax, (name, points) in zip(axes, [('noisy ring', noisy_ring),
                                     ('filled disk sample', noisy_disk)]):
    ax.scatter(points[:, 0], points[:, 1], s=18)
    ax.set(title=name, aspect='equal', xlim=(-1.2, 1.2), ylim=(-1.2, 1.2))
plt.show()

## Demonstration 4, alongside slide “A first transparent diagnostic”

At threshold $\varepsilon$, join points at distance at most $\varepsilon$. For this graph,

$$\text{cycle rank}=E-V+C.$$

**Ask before running:** As $\varepsilon$ increases, should components and graph cycle rank move in the same direction?

In [ ]:
epsilons = np.linspace(0.08, 0.65, 24)
results = {}
for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
    results[name] = [threshold_graph_statistics(points, epsilon)
                     for epsilon in epsilons]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
for name, records in results.items():
    axes[0].plot(epsilons, [r['components'] for r in records], label=name)
    axes[1].plot(epsilons, [r['graph_cycle_rank'] for r in records], label=name)
axes[0].set(xlabel=r'threshold $\varepsilon$', ylabel='connected components')
axes[1].set(xlabel=r'threshold $\varepsilon$', ylabel=r'graph cycle rank $E-V+C$')
for ax in axes:
    ax.legend()
plt.show()

chosen_epsilon = 0.28
for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
    print(name, threshold_graph_statistics(points, chosen_epsilon))

### Reveal and pause

At small thresholds both graphs fragment. Components merge as the threshold grows. Graph cycle rank can then grow rapidly because new edges create graph cycles.

The rapid growth in graph cycle rank is the point at which to separate the threshold graph from its clique complex. Filled triangles would turn many three-edge graph cycles into boundaries, so the plotted graph-cycle-rank curve is not a curve of first Betti numbers for the full Rips complex.

## Demonstration 5, alongside slide “One threshold is a modelling choice”

Use three thresholds on exactly the same observations. The point is not to select a winner. It is to reveal how strongly a conclusion can depend on one threshold.

In [ ]:
for epsilon in [0.16, 0.28, 0.48]:
    print(f'epsilon = {epsilon:.2f}')
    for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
        print(' ', name, threshold_graph_statistics(points, epsilon))

### Teaching point

One threshold produces one constructed graph and one answer. A sweep makes the scale dependence visible, but graph cycle rank still belongs to a graph rather than to the filled simplicial complexes introduced in Week 2. The lesson here is that the representation and threshold are part of the result.

## Close the lecture: what has and has not been established

- Matching means, standard deviations and correlation do not imply matching spatial organisation.
- The observed coordinates, distance rule, threshold graph and any later complex are distinct objects.
- Homeomorphic continuous shapes need not have identical metric constructions at a fixed threshold.
- Graph cycle rank is not generally $H_1$ of the clique complex.
- Sweeping a threshold reveals sensitivity, but it does not by itself define simplicial homology.

Return to the slide “Before we leave” and ask the class to name one choice at every arrow:

$$\text{data}\to\text{representation}\to\text{metric}\to\text{complex}\to\text{summary}.$$